# Scientist #7: Christiaan Huygens — Wave Theory of Light (30s Blender 4.2 Cycles Dual-GPU Short)

**Full 30-Second Production Pipeline with Pre-Mastered Kokoro Voiceover, 116 BPM Baroque Score, 24 Spatial Foley SFX & Word-Synced Boxless Subtitles!**

### Quick Kaggle Setup (`GPU T4 x2` -> ~9-11 min total!):
1. Right panel -> **Session options** -> **Accelerator: `GPU T4 x2`** & **Internet: `ON`**.
2. Click **Run All**.
3. Final `1080x1920` H.264 MP4 saves directly to `/kaggle/working/christiaan_huygens_wave_light_short.mp4` for 1-tap download!

In [ ]:
# 1. Verify NVIDIA GPU(s) & Clone Complete Repo (includes mastered 30s audio + synced subtitles)
!nvidia-smi
!rm -rf huygens-wave-light-3d
!git clone https://github.com/ssghtc002-sudo/huygens-wave-light-3d.git
%cd huygens-wave-light-3d
!ls -lh audio/output/master_audio.wav subtitles/subtitles.ass

In [ ]:
# 2. Download & Extract Blender 4.2.3 LTS Headless (~25 sec, cached in /tmp/blender)
import os
if not os.path.exists('/tmp/blender/blender'):
    !wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /tmp/blender.tar.xz
    !mkdir -p /tmp/blender && tar -xf /tmp/blender.tar.xz -C /tmp/blender --strip-components=1
!/tmp/blender/blender --version

In [ ]:
# 3. Fast 10-Second 6-Beat Visual QA Preview (renders 1 hero frame per Beat & displays 2x3 grid!)
!CUDA_VISIBLE_DEVICES=0 /tmp/blender/blender --background --python src/build_huygens_scene.py -- --qa-only --samples 16

import glob, os, base64
from IPython.display import display, HTML

imgs = sorted(glob.glob('qa_frames/qa_beat*.png'))
cards = []
for p in imgs:
    b64 = base64.b64encode(open(p, 'rb').read()).decode()
    cards.append(f'<div style="display:inline-block;margin:6px;text-align:center;"><img src="data:image/png;base64,{b64}" width="210"/><br><small>{os.path.basename(p)}</small></div>')
display(HTML(''.join(cards)))

In [ ]:
# 4. Parallel Dual-GPU Production Render (900 frames @ 1080x1920, 16 samples + OPTIX + FAST GPU Denoise)
# Automatically splits 1..450 on GPU 0 and 451..900 on GPU 1 in parallel (~9-11 min on Kaggle T4 x2)!
# Resume-safe: skips existing PNGs if re-run.
import subprocess, time, glob

gpu_count = int(subprocess.check_output('nvidia-smi --list-gpus | wc -l', shell=True).decode().strip())
print(f'Detected {gpu_count} NVIDIA GPU(s)! Starting Cycles production render...')
t0 = time.time()

if gpu_count >= 2:
    p0 = subprocess.Popen(
        'CUDA_VISIBLE_DEVICES=0 /tmp/blender/blender --background --python src/build_huygens_scene.py -- --start-frame 1 --end-frame 450 --samples 16',
        shell=True, stdout=open('/tmp/gpu0.log', 'w'), stderr=subprocess.STDOUT
    )
    p1 = subprocess.Popen(
        'CUDA_VISIBLE_DEVICES=1 /tmp/blender/blender --background --python src/build_huygens_scene.py -- --start-frame 451 --end-frame 900 --samples 16',
        shell=True, stdout=open('/tmp/gpu1.log', 'w'), stderr=subprocess.STDOUT
    )
    while p0.poll() is None or p1.poll() is None:
        done = len(glob.glob('render_frames/frame_*.png'))
        elapsed = time.time() - t0
        rate = done / max(1.0, elapsed)
        eta = (900 - done) / max(0.01, rate) if done > 2 else 0
        print(f'\r[Dual-T4 Progress] {done}/900 frames ({done/9:.1f}%) | Elapsed: {elapsed/60:.1f}m | ETA: {eta/60:.1f}m   ', end='', flush=True)
        time.sleep(5)
    print()
    if p0.returncode != 0 or p1.returncode != 0:
        print('--- GPU 0 LOG TAIL ---\n' + ''.join(open('/tmp/gpu0.log').readlines()[-25:]))
        print('--- GPU 1 LOG TAIL ---\n' + ''.join(open('/tmp/gpu1.log').readlines()[-25:]))
        raise RuntimeError(f'Worker error: p0={p0.returncode} p1={p1.returncode}')
else:
    ! /tmp/blender/blender --background --python src/build_huygens_scene.py -- --start-frame 1 --end-frame 900 --samples 16

rendered = len(glob.glob('render_frames/frame_*.png'))
print(f'Render complete: {rendered}/900 frames in {(time.time()-t0)/60:.1f} min!')

In [ ]:
# 5. Mux 900 Frames + Mastered 30s Kokoro Audio + Word-Synced Boxless Subtitles -> Final 1080x1920 MP4!
!mkdir -p output
!ffmpeg -y -framerate 30 -i render_frames/frame_%04d.png -i audio/output/master_audio.wav \
  -vf "ass=subtitles/subtitles.ass" -c:v libx264 -profile:v high -crf 16 -pix_fmt yuv420p \
  -c:a aac -b:a 192k -shortest -movflags +faststart output/christiaan_huygens_wave_light_short.mp4 -loglevel warning

# Copy directly to /kaggle/working/ so it appears immediately in Kaggle Output panel for 1-tap download!
import os, shutil
out_mp4 = 'output/christiaan_huygens_wave_light_short.mp4'
if os.path.isdir('/kaggle/working'):
    shutil.copy2(out_mp4, '/kaggle/working/christiaan_huygens_wave_light_short.mp4')
    print('Copied to /kaggle/working/christiaan_huygens_wave_light_short.mp4 — Ready in Kaggle Output Panel!')

!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,nb_frames,duration -of compact output/christiaan_huygens_wave_light_short.mp4
!ls -lh output/christiaan_huygens_wave_light_short.mp4

try:
    from google.colab import files
    files.download(out_mp4)
except ImportError:
    from IPython.display import Video, display
    display(Video(out_mp4, embed=False, width=320))